In [1]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU available:", gpus)
else:
    print("No GPU, using CPU")

print()


import os # Configure which GPU
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, ITURadioMaterial, SceneObject

import matplotlib.pyplot as plt

import mitsuba as mi

import numpy as np

# Import Sionna utils from the wireless class
try:
    import sionnautils
except ImportError as e:
    # Install Sionna if package is not already installed
    !pip install git+https://github.com/sdrangan/wirelesscomm.git
    import sionnautils
    
    
    
from scipy.spatial.transform import Rotation as R

import mitsuba as mi
import drjit as dr
from sionna.rt import AntennaPattern, PlanarArray, register_antenna_pattern
from sionnautils.custom_scene import list_scenes, get_scene


2026-01-14 16:16:28.122077: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-14 16:16:28.169524: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-14 16:16:29.497844: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



In [2]:
from sionnautils.custom_scene import list_scenes, get_scene
scenes = list_scenes()
print(scenes)

scene_path, map_data = get_scene('nyu_tandon')
for k, v in map_data.items():
    print(f'{k}: {v}')

scene = load_scene(scene_path,merge_shapes=True)

floor = scene.get('ground')
# print(f'Floor material: {floor.radio_material.name}')
floor.radio_material = ITURadioMaterial("itu_concrete",
                                "concrete",
                                thickness=0.01,
                                color=(0.5, 0.5, 0.5))

scene.remove("itu_wet_ground")

for name, obj in scene.objects.items():
    print(f'{name:<15}{obj.radio_material.name}')
# scene.render(camera=my_cam, num_samples=512)

scene.radio_materials

['nyu_tandon']
bbox_lat: [40.69012764197041, 40.699120858029595]
bbox_long: [-73.99156687083165, -73.97970552916836]
address: 5 MetroTech Center, Brooklyn, NY 11201
descr: NYU Tandon campus
2026-01-14 16:16:32 WARN  [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).
no-name-1      itu_marble
ground         itu_concrete


{'itu_marble': ITURadioMaterial type=marble
                  eta_r=7.074
                  sigma=0.018
                  thickness=0.100
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000,
 'itu_concrete': ITURadioMaterial type=concrete
                  eta_r=5.240
                  sigma=0.123
                  thickness=0.010
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000}

In [3]:
import yaml

with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# cfg

In [4]:
from sionnautils.custom_scene import list_scenes, get_scene
scenes = list_scenes()
print(scenes)

scene_path, map_data = get_scene('nyu_tandon')
for k, v in map_data.items():
    print(f'{k}: {v}')

scene = load_scene(scene_path,merge_shapes=True)

floor = scene.get('ground')
# print(f'Floor material: {floor.radio_material.name}')
floor.radio_material = ITURadioMaterial("itu_concrete",
                                "concrete",
                                thickness=0.01,
                                color=(0.5, 0.5, 0.5))

scene.remove("itu_wet_ground")

for name, obj in scene.objects.items():
    print(f'{name:<15}{obj.radio_material.name}')
# scene.render(camera=my_cam, num_samples=512)

scene.radio_materials

['nyu_tandon']
bbox_lat: [40.69012764197041, 40.699120858029595]
bbox_long: [-73.99156687083165, -73.97970552916836]
address: 5 MetroTech Center, Brooklyn, NY 11201
descr: NYU Tandon campus
2026-01-14 16:16:33 WARN  [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).
no-name-2      itu_marble
ground         itu_concrete


{'itu_marble': ITURadioMaterial type=marble
                  eta_r=7.074
                  sigma=0.018
                  thickness=0.100
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000,
 'itu_concrete': ITURadioMaterial type=concrete
                  eta_r=5.240
                  sigma=0.123
                  thickness=0.010
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000}

In [5]:
class Parch_Pattern(AntennaPattern):
    def vertical_cut(self, theta: mi.Float) -> mi.Float:
        theta_3dB = dr.deg2rad(125.0)
        SLA_v = 22.5
        return -dr.minimum(12 * dr.square((theta - dr.deg2rad(90)) / theta_3dB), SLA_v)

    def horizontal_cut(self, phi: mi.Float) -> mi.Float:
        phi_3dB = dr.deg2rad(125.0)
        A_max = 22.5
        return -dr.minimum(12 * dr.square(phi / phi_3dB), A_max)

    def combined_pattern(self, theta: mi.Float, phi: mi.Float) -> mi.Float:
        A_max = 22.5
        a_v = self.vertical_cut(theta)
        a_h = self.horizontal_cut(phi)
        total = a_v + a_h
        return -dr.minimum(-total, A_max)

    def __init__(self, polarization: str = "V"):
        if polarization not in {"V", "H", "VH"}:
            raise ValueError("Polarization must be 'V', 'H', or 'VH'")
        self.polarization = polarization

        def my_pattern(theta, phi):
            gain_dB = self.combined_pattern(theta, phi)
            gain_linear = dr.power(10.0, gain_dB / 20.0)

            if self.polarization == "V":
                c_theta = mi.Complex2f(gain_linear, dr.zeros(mi.Float, dr.width(theta)))
                c_phi = mi.Complex2f(dr.zeros(mi.Float, dr.width(phi)), dr.zeros(mi.Float, dr.width(phi)))
            elif self.polarization == "H":
                c_theta = mi.Complex2f(dr.zeros(mi.Float, dr.width(theta)), dr.zeros(mi.Float, dr.width(theta)))
                c_phi = mi.Complex2f(gain_linear, dr.zeros(mi.Float, dr.width(phi)))
            else:  # "VH"
                scale = gain_linear / dr.sqrt(2.0)
                c_theta = mi.Complex2f(scale, dr.zeros(mi.Float, dr.width(theta)))
                c_phi = mi.Complex2f(scale, dr.zeros(mi.Float, dr.width(phi)))

            return c_theta, c_phi

        self.patterns = [lambda theta, phi: my_pattern(theta, phi)]
        
desired_pol = "V"
register_antenna_pattern("patch", lambda: Parch_Pattern(desired_pol))

In [6]:
cfg["tx"]["n_tx"]

2

In [7]:
for index_tx in range(cfg["tx"]["n_tx"]):

    scene.tx_array = PlanarArray(num_rows=cfg["tx"]["array_size"][index_tx][0],
                             num_cols=cfg["tx"]["array_size"][index_tx][1],
                             vertical_spacing=cfg["tx"]["vertical_spacing"][index_tx],
                             horizontal_spacing=cfg["tx"]["horizontal_spacing"][index_tx],
                             pattern=cfg["tx"]["pattern"][index_tx],
                             polarization=cfg["tx"]["polarization"][index_tx])
    tx = Transmitter(name="tx-" + str(index_tx),
                            position=cfg["tx"]["pos"][index_tx],
                            display_radius=cfg["tx"]["display_radius"][index_tx])

    # Add transmitter instance to scene
    scene.remove("tx-" + str(index_tx))
    scene.add(tx)

    tx.look_at(cfg["tx"]["look_at_pos"][index_tx])

In [9]:
from UE_config_3 import UE_v3


n_ue = cfg["ue"]["n_ue"]
ue_list = []

for index_ue in range(cfg["ue"]["n_ue"]):
    
    ue = UE_v3(scene = scene, id_ue= index_ue, cfg = cfg)
    ue_list.append(ue)
    
    
ue_list[0].rx_loc_pos_list

[[-290, 50, 100]]


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[array([-7.2, 15. ,  0. ]),
 array([  7.2, -15. ,   0. ]),
 array([7.2, 0. , 0. ]),
 array([-7.2,  0. ,  0. ])]

In [10]:
ue_list[0].set_orientation(0, np.pi / 2, 0)

[[-580, 100, 200]]


In [12]:
scene.rx_array = PlanarArray(num_rows=1,
                                        num_cols=1,
                                        vertical_spacing=0.5,
                                        horizontal_spacing=0.5,
                                        pattern=cfg["ue"]["rx_pattern"][3],
                                        polarization="V")

In [13]:
p_solver  = PathSolver()

paths_1 = p_solver(scene=scene,
                     max_depth=1,
                     los=True,
                     specular_reflection=True,
                     diffraction=True,
                     edge_diffraction=False,
                     refraction=True,
                     diffuse_reflection=True)





In [14]:
scene.preview(paths=paths_1)

In [15]:
paths_2 = p_solver(scene=scene,
                     max_depth=1,
                     los=True,
                     specular_reflection=True,
                     diffraction=False,
                     edge_diffraction=False,
                     refraction=True,
                     diffuse_reflection=True)



scene.preview(paths=paths_1)